# Stack rasters and extract tables with WGS84 coordinates

This script loops over years, loads all spatial data files (NPP, climate yearly, climate vegetation period, elevation, region) for a year, stacks them and extracts a table for each year. The table contains a column for each raster band, WGS84 coordinates of the MODIS pixel centers, the year, and a row for each pixel that does not have a missing value for NPP. The data is exported as a parquet file.

## 1. Load and inspect data

In [32]:
# Import necessary libraries
from pathlib import Path

import numpy as np
import polars as pl
import geopandas as gpd
import rasterio
from rasterio.windows import Window
from rasterio.transform import xy

from pyproj import Transformer

In [33]:
# Define years to process
YEARS = range(2002, 2024)

# Define paths to data
# path = Path("/Volumes/TOSHIBA EXT/non-equilibrium/data")
path = Path("/Users/Wanja/Documents/non-equilibrium_data")
climate_path = Path(path, "climate_yearly_new")
npp_path = Path(path, "npp")
elevation_path = Path(path, "elevation/Elevation.tif")
regions_path = Path(path, "world-administrative-boundaries/world-administrative-boundaries.shp")
output_path = Path(path, "tables_modis_new")

In [34]:
# Inspect raster files 
files = [
    climate_path / "Climate_yearly_2002.tif",
    climate_path / "Climate_vegetation_period_2002.tif",
    npp_path / "NPP_2002-01-01.tif",
    elevation_path,
]

for file in files:
    with rasterio.open(file) as src:
        print("=" * 60)
        print(file.name)
        print("=" * 60)
        print(f"Bands: {src.count}")
        print(f"CRS: {src.crs}")
        print(f"Resolution: {src.res}")
        print(f"Nodata: {src.nodata}")
        print()

        for i, desc in enumerate(src.descriptions, start=1):
            print(f"Band {i}: {desc}")

Climate_yearly_2002.tif
Bands: 7
CRS: PROJCS["MODIS Sinusoidal",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Sinusoidal"],PARAMETER["longitude_of_center",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Resolution: (5000.0, 5000.0)
Nodata: nan

Band 1: tmmn_mean
Band 2: tmmx_mean
Band 3: tmmn_min
Band 4: tmmn_max
Band 5: tmmx_min
Band 6: tmmx_max
Band 7: pr_sum
Climate_vegetation_period_2002.tif
Bands: 8
CRS: PROJCS["MODIS Sinusoidal",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Sinusoidal"],PARAMETER["longitude_of_center",0],PARAMETER["false_east

In [35]:
npp_file = npp_path / "NPP_2002-01-01.tif"

with rasterio.open(npp_file) as src:

    npp_band2 = src.read(2)

    if src.nodata is not None:
        valid = npp_band2 != src.nodata
    else:
        valid = ~np.isnan(npp_band2)

    print("Raster shape:", npp_band2.shape)
    print("Total pixels:", npp_band2.size)
    print("Valid pixels:", valid.sum())
    print("Missing pixels:", (~valid).sum())
    print("Percentage valid:", valid.mean() * 100)

Raster shape: (4004, 8008)
Total pixels: 32064032
Valid pixels: 2913692
Missing pixels: 29150340
Percentage valid: 9.087104204486822


/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_59091/1365574156.py:5: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  npp_band2 = src.read(2)


In [36]:
# Load regions shapefile
regions = gpd.read_file(regions_path)

regions = regions[[
    "iso3",
    "name",
    "continent",
    "region",
    "geometry"
]]

In [37]:
# Inspect regions GeoDataFrame
print("=" * 60)
print("World administrative boundaries")
print("=" * 60)

print(f"Number of regions: {len(regions)}")
print(f"CRS: {regions.crs}")
print(f"Geometry types:")
print(regions.geometry.geom_type.value_counts())

print("\nColumns:")
print(regions.columns.tolist())

print("\nFirst rows:")
display(regions.head())

World administrative boundaries
Number of regions: 256
CRS: EPSG:4326
Geometry types:
Polygon         141
MultiPolygon    115
Name: count, dtype: int64

Columns:
['iso3', 'name', 'continent', 'region', 'geometry']

First rows:


,iso3,name,continent,region,geometry
0,UGA,Uganda,Africa,Eastern Africa,"POLYGON ((33.9211 -1.00194, 33.92027 -1.00111,..."
1,UZB,Uzbekistan,Asia,Central Asia,"POLYGON ((70.97081 42.25467, 70.98054 42.26205..."
2,IRL,Ireland,Europe,Northern Europe,"MULTIPOLYGON (((-9.97014 54.02083, -9.93833 53..."
3,ERI,Eritrea,Africa,Eastern Africa,"MULTIPOLYGON (((40.13583 15.7525, 40.12861 15...."
4,NaN,Ma'tan al-Sarra,Africa,Northern Africa,"POLYGON ((33.25104 21.99977, 34.15064 21.99603..."


In [38]:
# Define a transformer to convert from Sinusoidal projection to WGS84 (EPSG:4326)
transformer = Transformer.from_crs(
    "+proj=sinu +R=6371007.181",
    "EPSG:4326",
    always_xy=True
)

## 2. Functions to stack rasters and extract tables with WGS84 coordinates

In [39]:
def generate_windows(src):
    """
    Generate raster block windows for block-wise processing. 
    Windows are generated based on the block shapes of the raster dataset (256x256 pixels).

    Parameters
    ----------
    src : rasterio.io.DatasetReader
        Open raster dataset.

    Yields
    ------
    rasterio.windows.Window
        Window covering one internal raster block.
    """

    block_h, block_w = src.block_shapes[0]

    for row in range(0, src.height, block_h):
        for col in range(0, src.width, block_w):

            yield Window(
                col,
                row,
                min(block_w, src.width-col),
                min(block_h, src.height-row)
            )

In [40]:
def window_coordinates(src, window):
    """
    Compute WGS84 coordinates for the center of each pixel in a raster window.

    Parameters
    ----------
    src : rasterio.io.DatasetReader
        Open raster dataset.
    window : rasterio.windows.Window
        Raster window to process.

    Returns
    -------
    tuple of numpy.ndarray
        Longitude and latitude arrays with shape (height, width).
    """

    rows, cols = np.meshgrid(
        np.arange(
            window.row_off,
            window.row_off + window.height
        ),
        np.arange(
            window.col_off,
            window.col_off + window.width
        ),
        indexing="ij"
    )

    xs, ys = rasterio.transform.xy(
        src.transform,
        rows,
        cols,
        offset="center"
    )

    xs = np.asarray(xs)
    ys = np.asarray(ys)

    lon, lat = transformer.transform(xs, ys)
    lon = np.array(lon).reshape(window.height, window.width)
    lat = np.array(lat).reshape(window.height, window.width)

    return np.asarray(lon), np.asarray(lat)

In [41]:
def process_block(
    climate_yearly,
    climate_veg,
    npp,
    elevation,
    window,
    year
):
    """
    Extract valid pixels from a raster window and return them as a Polars DataFrame.

    Reads all climate and NPP bands, the mean elevation band, and pixel
    coordinates. Pixels are filtered based on valid values in the selected
    NPP band, preserving raster band descriptions as column names.

    Parameters
    ----------
    climate_yearly : rasterio.io.DatasetReader
        Open yearly climate raster.
    climate_veg : rasterio.io.DatasetReader
        Open vegetation-period climate raster.
    npp : rasterio.io.DatasetReader
        Open NPP raster.
    elevation : rasterio.io.DatasetReader
        Open elevation raster.
    window : rasterio.windows.Window
        Raster window to process.
    year : int
        Year associated with the raster data.

    Returns
    -------
    polars.DataFrame or None
        Table containing all valid pixels in the window, or None if no valid
        NPP pixels are present.
    """

    # Read raster blocks
    yearly = climate_yearly.read(window=window)
    veg = climate_veg.read(window=window)

    npp_data = npp.read(window=window)
    elevation_data = elevation.read(1, window=window)

    # Convert elevation nodata to NaN
    elevation_data = elevation_data.astype("float32")
    elevation_data[elevation_data == -9999] = np.nan

    # Coordinates
    lon, lat = window_coordinates(
        npp,
        window
    )

    # --------------------------------------------------
    # Mask: keep only pixels with valid NPP
    # --------------------------------------------------

    if npp.nodata is not None:
        mask = npp_data[1] != npp.nodata
    else:
        mask = ~np.isnan(npp_data[1])

    if mask.sum() == 0:
        return None
        

    # --------------------------------------------------
    # Create output dictionary
    # --------------------------------------------------

    data = {}


    # --------------------------------------------------
    # Climate yearly bands
    # --------------------------------------------------

    for i, name in enumerate(climate_yearly.descriptions):

        if name is None:
            name = f"climate_yearly_band_{i+1}"

        data[name] = yearly[i][mask]


    # --------------------------------------------------
    # Climate vegetation period bands
    # --------------------------------------------------

    for i, name in enumerate(climate_veg.descriptions):

        if name is None:
            name = f"climate_vegetation_period_band_{i+1}"

        data[name] = veg[i][mask]


    # --------------------------------------------------
    # NPP bands
    # --------------------------------------------------

    for i, name in enumerate(npp.descriptions):

        if name is None:
            name = f"npp_band_{i+1}"

        data[name] = npp_data[i][mask]


    # --------------------------------------------------
    # Elevation
    # --------------------------------------------------

    # only first band
    data["elevation_mean"] = elevation_data[mask]


    # --------------------------------------------------
    # Coordinates and year
    # --------------------------------------------------

    data["longitude"] = lon[mask]
    data["latitude"] = lat[mask]
    data["year"] = np.full(mask.sum(), year)


    # --------------------------------------------------
    # Create table
    # --------------------------------------------------

    return pl.DataFrame(data)

In [42]:
def add_regions(df):
    """
    Assign administrative regions to pixel coordinates using a spatial join. 
    If one pixel falls within multiple regions, only the first match is kept.

    Parameters
    ----------
    df : polars.DataFrame
        Table containing longitude and latitude columns in WGS84.

    Returns
    -------
    polars.DataFrame
        Input table with administrative region attributes appended.
    """

    gdf = gpd.GeoDataFrame(
        df.to_pandas(),
        geometry=gpd.points_from_xy(
            df["longitude"],
            df["latitude"]
        ),
        crs="EPSG:4326"
    )
    
    joined = gpd.sjoin(
        gdf,
        regions,
        predicate="within",
        how="left"
    )
    
    # A point should normally belong to only one region.
    # Keep the first match in case of overlapping polygons.
    joined = (
        joined
        .drop_duplicates(
            subset=["longitude", "latitude"]
        )
    )

    joined = joined.drop(columns=["geometry", "index_right"])

    return pl.from_pandas(joined)

In [43]:
def process_year(year):
    """
    Process all raster data for a single year and export pixel-level data.

    Reads yearly climate, vegetation-period climate, NPP, and elevation rasters,
    processes them block-wise to extract valid NPP pixels, assigns administrative
    regions, and saves the resulting table as a Parquet file.

    Parameters
    ----------
    year : int
        Year to process.

    Returns
    -------
    None
        Writes a yearly Parquet file to the output directory.
    """
        
    climate_yearly = climate_path / f"Climate_yearly_{year}.tif"

    climate_veg = climate_path / f"Climate_vegetation_period_{year}.tif"

    npp_file = npp_path / f"NPP_{year}-01-01.tif"

    with rasterio.open(climate_yearly) as yearly,\
         rasterio.open(climate_veg) as veg,\
         rasterio.open(npp_file) as npp,\
         rasterio.open(elevation_path) as elevation:

        tables = []

        for window in generate_windows(npp):

            df = process_block(
                yearly,
                veg,
                npp,
                elevation,
                window,
                year
            )

            if df is None:
                continue

            df = add_regions(df)

            tables.append(df)

    yearly_table = pl.concat(tables)

    yearly_table.write_parquet(
        output_path / f"table_modis_{year}.parquet"
    )

## 3. Main

In [44]:
for year in YEARS:

    print(year)

    process_year(year)

/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_59091/2562000295.py:43: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  elevation_data = elevation.read(1, window=window)


2002


/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_59091/2562000295.py:43: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  elevation_data = elevation.read(1, window=window)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_59091/2562000295.py:43: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  elevation_data = elevation.read(1, window=window)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_59091/2562000295.py:43: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  elevation_data = elevation.read(1, window=window)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipyker

2003


/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_59091/2562000295.py:43: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  elevation_data = elevation.read(1, window=window)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_59091/2562000295.py:43: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  elevation_data = elevation.read(1, window=window)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_59091/2562000295.py:43: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  elevation_data = elevation.read(1, window=window)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipyker

...

/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_59091/2562000295.py:43: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  elevation_data = elevation.read(1, window=window)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_59091/2562000295.py:43: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  elevation_data = elevation.read(1, window=window)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_59091/2562000295.py:43: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  elevation_data = elevation.read(1, window=window)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipyker

2022


/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_59091/2562000295.py:43: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  elevation_data = elevation.read(1, window=window)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_59091/2562000295.py:43: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  elevation_data = elevation.read(1, window=window)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_59091/2562000295.py:43: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  elevation_data = elevation.read(1, window=window)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipyker

2023


/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_59091/2562000295.py:43: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  elevation_data = elevation.read(1, window=window)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_59091/2562000295.py:43: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  elevation_data = elevation.read(1, window=window)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_59091/2562000295.py:43: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  elevation_data = elevation.read(1, window=window)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipyker